# 2. Tiny Overfit

This notebook runs the tiny overfit pytest check and then launches short baseline and AttnRes training runs on the debug config.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/AtinChing/AttnResGPT-mini.git'
REPO_NAME = 'AttnResGPT-mini'

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

candidates = [Path(f'/content/{REPO_NAME}'), Path(f'/content/drive/MyDrive/{REPO_NAME}'), Path.cwd()]
repo_root = next((p for p in candidates if (p / 'requirements.txt').exists() and (p / 'src').exists()), None)

if repo_root is None:
    target = Path(f'/content/{REPO_NAME}')
    print(f'Cloning {REPO_URL} into {target} ...')
    subprocess.run(['git', 'clone', REPO_URL, str(target)], check=True)
    repo_root = target
else:
    print(f'Using existing repo at {repo_root}')

%cd {repo_root}
!pip -q install -r requirements.txt

In [ ]:
import torch
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_name:', torch.cuda.get_device_name(0))

In [ ]:
!pytest -q tests/test_tiny_overfit.py -m slow

In [ ]:
!python -m src.train --config configs/debug_tiny.yaml --overrides experiment.name=nb2_overfit_baseline model.architecture=baseline training.max_steps=150 training.eval_interval=50 training.checkpoint_interval=150
!python -m src.train --config configs/debug_tiny.yaml --overrides experiment.name=nb2_overfit_attnres model.architecture=attnres model.attnres.enabled=true training.max_steps=150 training.eval_interval=50 training.checkpoint_interval=150

In [ ]:
import json
from pathlib import Path

for pattern in ['nb2_overfit_baseline_*', 'nb2_overfit_attnres_*']:
    run_dir = sorted(Path('runs').glob(pattern))[-1]
    summary = json.loads((run_dir / 'run_summary.json').read_text())
    print(run_dir.name)
    print({k: summary[k] for k in ['val_loss', 'val_perplexity', 'best_val_loss'] if k in summary})
